# Amine Representation Comparison for BSH Activity Prediction

The current BSH model uses 1024-bit Morgan fingerprints to represent only 26 amines.
This creates a **97% sparse** feature block occupying half the feature space, preventing
the model from learning meaningful enzyme-amine interactions. Log loss curves show
train/val divergence — the model memorizes rather than learns.

**Goal:** Compare 7 amine representations to find one that enables genuine learning:

| Code | Representation | Dims | Rationale |
|------|---------------|------|----------|
| A | Morgan FP 1024-bit | 1024 | Current baseline |
| B | Morgan FP 256-bit | 256 | Less sparse (75% info retained) |
| C | MACCS keys | 167 | Predefined structural patterns |
| D | Physicochemical descriptors | ~15 | MW, LogP, TPSA, etc. Dense, interpretable |
| E | One-hot encoding | 26 | Let model learn per-amine biases directly |
| F | Physicochemical + one-hot | ~41 | Chemical similarity + amine identity |
| G | PCA enzyme x physicochemical interaction | ~1615 | Cross-features for interaction learning |

**Enzyme representation:** noncons_mean (conservation < 0.5, mean pooled, 1024-dim)

**Key diagnostic:** Train AND validation log loss curves — do they converge?

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors

from Bio import SeqIO

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

AMINE_REPR_DIR = OUTPUT_DIR / "model_outputs" / "amine_representation"
AMINE_REPR_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

N_SPLITS = 10
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

print("Setup complete.")
print(f"Output directory: {AMINE_REPR_DIR}")

## 2. Load Data

In [ ]:
# --- Per-residue ProtT5 embeddings ---
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"
per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        per_residue_embeddings[uniprot_id] = f[key][:]
print(f"Per-residue embeddings: {len(per_residue_embeddings)} enzymes")

# --- Conservation scores ---
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
core_mask = df_cons['gap_fraction'] < 0.5
df_core = df_cons[core_mask].copy()
print(f"Core alignment positions (gap < 50%): {len(df_core)}")

# --- Alignment mapping ---
alignment_to_seq = {}
for record in SeqIO.parse(OUTPUT_DIR / "bsh_aligned.fasta", 'fasta'):
    parts = record.id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else record.id
    seq = str(record.seq)
    mapping = {}
    seq_pos = 0
    for aln_pos, char in enumerate(seq):
        if char != '-':
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    alignment_to_seq[uniprot_id] = mapping
overlap = set(alignment_to_seq.keys()) & set(per_residue_embeddings.keys())
print(f"Enzymes with alignment + embeddings: {len(overlap)}")

# --- Activity labels ---
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']
df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]
df_agg = df_activity.groupby(['Enzyme', 'Amine']).agg(
    active=('active_approach2', 'any'),
    n_products=('ProductName', 'count'),
    n_active_products=('active_approach2', 'sum')
).reset_index()
print(f"Activity data: {df_agg.shape[0]} enzyme-amine pairs")
print(f"  Active: {df_agg['active'].sum()}, Inactive: {(~df_agg['active']).sum()} ({df_agg['active'].mean():.1%} active)")

# --- Amine SMILES ---
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")

name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

# Parse all SMILES into RDKit molecules
amine_mols = {}  # normalized_name -> mol
amine_smiles_map = {}  # normalized_name -> smiles
for _, row in df_smiles.iterrows():
    name = row['Compound_Name']
    smiles = row['SMILES']
    norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
    if pd.isna(smiles):
        continue
    smiles_clean = smiles.split('.')[0]
    mol = Chem.MolFromSmiles(smiles_clean)
    if mol is not None:
        amine_mols[norm_name] = mol
        amine_smiles_map[norm_name] = smiles_clean

amines_needed = df_agg['Amine'].unique()
print(f"Amines needed: {len(amines_needed)}, parsed: {len(amine_mols)}")
print(f"Missing: {set(amines_needed) - set(amine_mols.keys())}")

In [ ]:
def get_nonconserved_embedding(enzyme_id, conservation_threshold=0.5, pooling='mean'):
    """Extract and pool per-residue embeddings at non-conserved positions."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    return selected.mean(axis=0)

# Precompute enzyme embeddings (noncons_mean, conservation < 0.5)
enzyme_embeddings = {}
for enz_id in overlap:
    emb = get_nonconserved_embedding(enz_id, 0.5, 'mean')
    if emb is not None:
        enzyme_embeddings[enz_id] = emb
print(f"Precomputed enzyme embeddings: {len(enzyme_embeddings)} (1024-dim each)")

## 3. Compute All 7 Amine Representations

In [ ]:
# Build a sorted, consistent amine list for one-hot encoding
all_amines_sorted = sorted(amines_needed)
amine_to_idx = {a: i for i, a in enumerate(all_amines_sorted)}
n_amines = len(all_amines_sorted)

# === Representation A: Morgan FP 1024-bit (baseline) ===
repr_A = {}
for name, mol in amine_mols.items():
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    repr_A[name] = np.array(fp, dtype=np.float32)
for a in amines_needed:
    if a not in repr_A:
        repr_A[a] = np.zeros(1024, dtype=np.float32)

# === Representation B: Morgan FP 256-bit ===
repr_B = {}
for name, mol in amine_mols.items():
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=256)
    repr_B[name] = np.array(fp, dtype=np.float32)
for a in amines_needed:
    if a not in repr_B:
        repr_B[a] = np.zeros(256, dtype=np.float32)

# === Representation C: MACCS keys (167-bit) ===
repr_C = {}
for name, mol in amine_mols.items():
    fp = MACCSkeys.GenMACCSKeys(mol)
    repr_C[name] = np.array(fp, dtype=np.float32)
for a in amines_needed:
    if a not in repr_C:
        repr_C[a] = np.zeros(167, dtype=np.float32)

# === Representation D: Physicochemical descriptors ===
def compute_physicochemical(mol):
    """Compute ~15 interpretable physicochemical descriptors."""
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol),
        rdMolDescriptors.CalcNumAmideBonds(mol),
        Descriptors.NumValenceElectrons(mol),
        Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
        Descriptors.BalabanJ(mol) if Descriptors.BalabanJ(mol) != 0 else 0.0,
    ], dtype=np.float32)

PHYSCHEM_NAMES = [
    'MolWt', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds', 'AromaticRings',
    'AliphaticRings', 'FractionCSP3', 'HeavyAtoms', 'AmideBonds',
    'ValenceElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'BalabanJ'
]
N_PHYSCHEM = len(PHYSCHEM_NAMES)

repr_D = {}
for name, mol in amine_mols.items():
    repr_D[name] = compute_physicochemical(mol)
for a in amines_needed:
    if a not in repr_D:
        repr_D[a] = np.zeros(N_PHYSCHEM, dtype=np.float32)

# === Representation E: One-hot encoding ===
repr_E = {}
for name in amines_needed:
    vec = np.zeros(n_amines, dtype=np.float32)
    if name in amine_to_idx:
        vec[amine_to_idx[name]] = 1.0
    repr_E[name] = vec

# === Representation F: Physicochemical + one-hot ===
repr_F = {}
for name in amines_needed:
    repr_F[name] = np.concatenate([repr_D.get(name, np.zeros(N_PHYSCHEM)), repr_E[name]])

# Print summary
representations = {
    'A_morgan1024': repr_A,
    'B_morgan256': repr_B,
    'C_maccs': repr_C,
    'D_physicochemical': repr_D,
    'E_onehot': repr_E,
    'F_physchem_onehot': repr_F,
}

print("Amine Representations:")
print(f"{'Code':<25} {'Dims':>6} {'Sparsity':>10}")
print("-" * 45)
for code, rep_dict in representations.items():
    sample = list(rep_dict.values())[0]
    dims = len(sample)
    # Compute average sparsity across all amines
    all_vecs = np.array([rep_dict[a] for a in amines_needed if a in rep_dict])
    sparsity = (all_vecs == 0).mean()
    print(f"{code:<25} {dims:>6} {sparsity:>9.1%}")

In [ ]:
# Show physicochemical descriptor values for all amines
physchem_data = []
for name in all_amines_sorted:
    if name in repr_D:
        vals = repr_D[name]
        row = {'amine': name}
        for i, pname in enumerate(PHYSCHEM_NAMES):
            row[pname] = vals[i]
        physchem_data.append(row)

df_physchem = pd.DataFrame(physchem_data).set_index('amine')
print("Physicochemical descriptors per amine:")
display(df_physchem.round(2))

## 4. Build Feature Matrices

In [ ]:
def build_feature_matrix(amine_repr_dict, repr_name):
    """
    Build X, y by concatenating enzyme embeddings + amine representation.
    Returns X, y, enzymes, amines lists.
    """
    X_list, y_list = [], []
    enzymes, amines = [], []
    skipped = 0
    for _, row in df_agg.iterrows():
        enzyme, amine = row['Enzyme'], row['Amine']
        if enzyme not in enzyme_embeddings:
            skipped += 1
            continue
        if amine not in amine_repr_dict:
            skipped += 1
            continue
        features = np.concatenate([enzyme_embeddings[enzyme], amine_repr_dict[amine]])
        X_list.append(features)
        y_list.append(int(row['active']))
        enzymes.append(enzyme)
        amines.append(amine)
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y, enzymes, amines

# Build feature matrices for representations A-F
feature_matrices = {}
for code, rep_dict in representations.items():
    X, y, enzymes, amines = build_feature_matrix(rep_dict, code)
    feature_matrices[code] = (X, y, enzymes, amines)
    amine_dims = len(list(rep_dict.values())[0])
    print(f"{code}: X={X.shape}, amine_dims={amine_dims}, enzyme_dims=1024, active={y.mean():.1%}")

# === Representation G: PCA enzyme x physicochemical interaction ===
# First, PCA reduce enzyme embeddings to 100 dims
# Collect all enzyme embeddings used in the dataset
ref_X, ref_y, ref_enzymes, ref_amines = feature_matrices['A_morgan1024']
enzyme_ids_in_data = sorted(set(ref_enzymes))
enzyme_emb_matrix = np.array([enzyme_embeddings[e] for e in enzyme_ids_in_data], dtype=np.float32)

pca_enzyme = PCA(n_components=100, random_state=RANDOM_STATE)
enzyme_pca_matrix = pca_enzyme.fit_transform(enzyme_emb_matrix)
enzyme_pca_dict = {e: enzyme_pca_matrix[i] for i, e in enumerate(enzyme_ids_in_data)}
print(f"\nPCA variance retained (100 components): {pca_enzyme.explained_variance_ratio_.sum():.1%}")

# Build representation G: enzyme_pca(100) concat amine_physchem(15) concat outer_product(100*15=1500)
X_G_list, y_G_list = [], []
enz_G, ami_G = [], []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    if enzyme not in enzyme_pca_dict or amine not in repr_D:
        continue
    enz_pca = enzyme_pca_dict[enzyme]   # (100,)
    ami_phys = repr_D[amine]             # (15,)
    interaction = np.outer(enz_pca, ami_phys).flatten()  # (1500,)
    features = np.concatenate([enz_pca, ami_phys, interaction])  # 100+15+1500=1615
    X_G_list.append(features)
    y_G_list.append(int(row['active']))
    enz_G.append(enzyme)
    ami_G.append(amine)

X_G = np.array(X_G_list, dtype=np.float32)
y_G = np.array(y_G_list, dtype=np.int32)
feature_matrices['G_pca_interaction'] = (X_G, y_G, enz_G, ami_G)
print(f"G_pca_interaction: X={X_G.shape}, active={y_G.mean():.1%}")

print(f"\nTotal representations: {len(feature_matrices)}")

## 5. Train with Log Loss Tracking

In [ ]:
def enzyme_holdout_split_seed(X, y, enzymes, amines, seed, test_size=0.2, val_size=0.2):
    """Enzyme hold-out split with a specific random seed."""
    enzymes_arr = np.array(enzymes)
    unique_enzymes = np.unique(enzymes_arr)
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=bins
    )
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=tv_bins
    )
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
    }


def train_xgb_with_logloss(split_data):
    """
    Train regularized XGBoost, tracking train+val log loss per round.
    Returns metrics dict and log loss curves.
    """
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)
    
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=n_neg / n_pos,
        reg_alpha=1.0, reg_lambda=5.0,
        subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5,
        random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    # Extract log loss curves
    evals = model.evals_result()
    train_logloss = evals['validation_0']['logloss']
    val_logloss = evals['validation_1']['logloss']
    
    # Test predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_val = model.predict_proba(X_val)[:, 1]
    
    metrics = {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_train, y_proba_train) - log_loss(y_val, y_proba_val),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
        'train_acc': model.score(X_train, y_train),
        'val_acc': model.score(X_val, y_val),
        'n_rounds': len(train_logloss),
    }
    
    return metrics, train_logloss, val_logloss, model

print("Training functions ready.")

In [ ]:
%%time

# Train all representations across 10 splits
all_results = []
all_logloss_curves = {}  # {repr_code: [(train_curve, val_curve), ...] per split}

repr_codes = list(feature_matrices.keys())

for code in repr_codes:
    X, y, enzymes, amines = feature_matrices[code]
    all_logloss_curves[code] = []
    
    for i, seed in enumerate(SEEDS):
        split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=seed)
        metrics, train_ll, val_ll, model = train_xgb_with_logloss(split)
        metrics['representation'] = code
        metrics['split_idx'] = i
        metrics['split_seed'] = seed
        all_results.append(metrics)
        all_logloss_curves[code].append((train_ll, val_ll))
    
    # Print progress
    code_results = [r for r in all_results if r['representation'] == code]
    roc_vals = [r['roc_auc'] for r in code_results]
    gap_vals = [r['logloss_gap'] for r in code_results]
    print(f"{code}: ROC-AUC={np.mean(roc_vals):.3f}+/-{np.std(roc_vals):.3f}, "
          f"LogLoss gap={np.mean(gap_vals):.3f}+/-{np.std(gap_vals):.3f}")

df_results = pd.DataFrame(all_results)
print(f"\nTotal experiments: {len(df_results)} ({len(repr_codes)} repr x {N_SPLITS} splits)")
print("Done!")

## 6. Log Loss Convergence Plots

In [ ]:
# 7-panel plot: train vs val log loss for each representation (10 splits overlaid)
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes_flat = axes.flat

repr_labels = {
    'A_morgan1024': 'A: Morgan 1024',
    'B_morgan256': 'B: Morgan 256',
    'C_maccs': 'C: MACCS 167',
    'D_physicochemical': 'D: Physicochemical 15',
    'E_onehot': 'E: One-hot 26',
    'F_physchem_onehot': 'F: Phys+OneHot 41',
    'G_pca_interaction': 'G: PCA Interaction 1615',
}

for idx, code in enumerate(repr_codes):
    ax = axes_flat[idx]
    curves = all_logloss_curves[code]
    
    for i, (train_ll, val_ll) in enumerate(curves):
        rounds = np.arange(len(train_ll))
        alpha = 0.3
        ax.plot(rounds, train_ll, color='#3498db', alpha=alpha, linewidth=0.8)
        ax.plot(rounds, val_ll, color='#e74c3c', alpha=alpha, linewidth=0.8)
    
    # Plot mean curves
    min_len = min(len(c[0]) for c in curves)
    train_mean = np.mean([c[0][:min_len] for c in curves], axis=0)
    val_mean = np.mean([c[1][:min_len] for c in curves], axis=0)
    rounds = np.arange(min_len)
    ax.plot(rounds, train_mean, color='#2980b9', linewidth=2.5, label='Train (mean)')
    ax.plot(rounds, val_mean, color='#c0392b', linewidth=2.5, label='Val (mean)')
    
    # Final gap
    final_gap = val_mean[-1] - train_mean[-1]
    ax.set_title(f"{repr_labels.get(code, code)}\nFinal gap: {final_gap:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Boosting Round', fontsize=9)
    ax.set_ylabel('Log Loss', fontsize=9)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

# Hide the 8th subplot
axes_flat[7].set_visible(False)

plt.suptitle('Log Loss Convergence: Train vs Validation (10 splits overlaid)', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(AMINE_REPR_DIR / 'logloss_convergence_all.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {AMINE_REPR_DIR / 'logloss_convergence_all.png'}")

## 7. Performance Comparison

In [ ]:
# Summary statistics per representation
summary_rows = []
for code in repr_codes:
    mask = df_results['representation'] == code
    df_sub = df_results[mask]
    X_shape = feature_matrices[code][0].shape
    amine_dims = X_shape[1] - 1024 if code != 'G_pca_interaction' else X_shape[1]
    
    row = {
        'representation': code,
        'total_dims': X_shape[1],
        'amine_dims': amine_dims,
        'roc_auc_mean': df_sub['roc_auc'].mean(),
        'roc_auc_std': df_sub['roc_auc'].std(),
        'pr_auc_mean': df_sub['pr_auc'].mean(),
        'pr_auc_std': df_sub['pr_auc'].std(),
        'f1_mean': df_sub['f1'].mean(),
        'f1_std': df_sub['f1'].std(),
        'logloss_gap_mean': df_sub['logloss_gap'].mean(),
        'logloss_gap_std': df_sub['logloss_gap'].std(),
        'val_logloss_mean': df_sub['val_logloss'].mean(),
        'train_logloss_mean': df_sub['train_logloss'].mean(),
        'train_acc_mean': df_sub['train_acc'].mean(),
        'val_acc_mean': df_sub['val_acc'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("Summary (mean +/- std over 10 splits):")
print("=" * 100)
print(f"{'Repr':<25} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12} {'LL Gap':>12} {'Val LL':>8}")
print("-" * 100)
for _, r in df_summary.iterrows():
    print(f"{r['representation']:<25} {r['total_dims']:>5.0f} "
          f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
          f"{r['pr_auc_mean']:.3f}+/-{r['pr_auc_std']:.3f} "
          f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f} "
          f"{r['logloss_gap_mean']:>+.3f}+/-{r['logloss_gap_std']:.3f} "
          f"{r['val_logloss_mean']:>.3f}")

In [ ]:
# Bar charts: ROC-AUC, PR-AUC, F1, and log loss gap side by side
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

short_labels = ['A: Morg1024', 'B: Morg256', 'C: MACCS', 'D: Phys', 'E: OneHot', 'F: P+OH', 'G: Interact']
x = np.arange(len(repr_codes))

metrics_plot = [
    ('roc_auc', 'ROC-AUC', '#3498db'),
    ('pr_auc', 'PR-AUC', '#2ecc71'),
    ('f1', 'F1 Score', '#e67e22'),
    ('logloss_gap', 'Log Loss Gap (train - val)', '#e74c3c'),
]

for ax, (metric, title, color) in zip(axes.flat, metrics_plot):
    means = df_summary[f'{metric}_mean'].values
    stds = df_summary[f'{metric}_std'].values
    
    bars = ax.bar(x, means, yerr=stds, color=color, alpha=0.7, 
                  edgecolor='black', linewidth=0.5, capsize=4)
    
    # Value labels
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontsize=8, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    if metric == 'logloss_gap':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    elif metric in ('roc_auc',):
        ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

plt.suptitle('Amine Representation Comparison (Regularized XGBoost, 10 splits)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(AMINE_REPR_DIR / 'representation_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {AMINE_REPR_DIR / 'representation_comparison_bars.png'}")

In [ ]:
# Box plots: variance across splits
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, metric, title in zip(axes, 
    ['roc_auc', 'pr_auc', 'f1', 'val_logloss'],
    ['ROC-AUC', 'PR-AUC', 'F1', 'Validation Log Loss']):
    
    data_to_plot = []
    for code in repr_codes:
        vals = df_results[df_results['representation'] == code][metric].values
        data_to_plot.append(vals)
    
    bp = ax.boxplot(data_to_plot, labels=short_labels, patch_artist=True)
    colors_box = plt.cm.Set2(np.linspace(0, 1, len(repr_codes)))
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticklabels(short_labels, rotation=40, ha='right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Distribution Across 10 Enzyme Hold-Out Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(AMINE_REPR_DIR / 'representation_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {AMINE_REPR_DIR / 'representation_boxplots.png'}")

## 8. Feature Importance Analysis

In [ ]:
# Train one model per representation on the first split to extract feature importances
importance_results = []

for code in repr_codes:
    X, y, enzymes, amines = feature_matrices[code]
    split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=42)
    
    X_train, y_train = split['X_train'], split['y_train']
    X_val, y_val = split['X_val'], split['y_val']
    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)
    
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=n_neg / n_pos,
        reg_alpha=1.0, reg_lambda=5.0,
        subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5,
        random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    imp = model.feature_importances_
    total_features = len(imp)
    
    if code == 'G_pca_interaction':
        # G: first 100 = enzyme PCA, next 15 = amine physchem, last 1500 = interaction
        enzyme_imp = imp[:100].sum()
        amine_imp = imp[100:115].sum()
        interaction_imp = imp[115:].sum()
        total_imp = imp.sum()
        importance_results.append({
            'representation': code,
            'enzyme_frac': enzyme_imp / total_imp,
            'amine_frac': amine_imp / total_imp,
            'interaction_frac': interaction_imp / total_imp,
            'enzyme_dims': 100,
            'amine_dims': 15,
        })
    else:
        # A-F: first 1024 = enzyme, rest = amine
        enzyme_imp = imp[:1024].sum()
        amine_imp = imp[1024:].sum()
        total_imp = imp.sum()
        amine_dims = total_features - 1024
        importance_results.append({
            'representation': code,
            'enzyme_frac': enzyme_imp / total_imp,
            'amine_frac': amine_imp / total_imp,
            'interaction_frac': 0.0,
            'enzyme_dims': 1024,
            'amine_dims': amine_dims,
        })

df_importance = pd.DataFrame(importance_results)
print("Feature Importance Fractions:")
print("=" * 80)
print(f"{'Repr':<25} {'Enz dims':>8} {'Ami dims':>8} {'Enz frac':>10} {'Ami frac':>10} {'Inter frac':>10}")
print("-" * 80)
for _, r in df_importance.iterrows():
    print(f"{r['representation']:<25} {r['enzyme_dims']:>8.0f} {r['amine_dims']:>8.0f} "
          f"{r['enzyme_frac']:>9.1%} {r['amine_frac']:>9.1%} {r['interaction_frac']:>9.1%}")

In [ ]:
# Stacked bar chart: fraction of importance from amine vs enzyme features
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(repr_codes))
width = 0.6

enzyme_fracs = df_importance['enzyme_frac'].values
amine_fracs = df_importance['amine_frac'].values
interaction_fracs = df_importance['interaction_frac'].values

ax.bar(x, enzyme_fracs, width, label='Enzyme features', color='#3498db', alpha=0.8)
ax.bar(x, amine_fracs, width, bottom=enzyme_fracs, label='Amine features', color='#e74c3c', alpha=0.8)
ax.bar(x, interaction_fracs, width, bottom=enzyme_fracs + amine_fracs, 
       label='Interaction features', color='#2ecc71', alpha=0.8)

# Annotate amine fraction
for i, (ef, af, inf) in enumerate(zip(enzyme_fracs, amine_fracs, interaction_fracs)):
    ax.text(i, ef + af/2, f'{af:.1%}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if inf > 0.01:
        ax.text(i, ef + af + inf/2, f'{inf:.1%}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Fraction of Total Feature Importance', fontsize=12)
ax.set_title('Feature Importance: Enzyme vs Amine vs Interaction', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(AMINE_REPR_DIR / 'amine_importance_fraction.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {AMINE_REPR_DIR / 'amine_importance_fraction.png'}")

In [ ]:
# Sparsity and descriptor overview
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Sparsity comparison
ax = axes[0]
sparsities = []
for code in repr_codes:
    if code == 'G_pca_interaction':
        # Interaction features are dense
        X = feature_matrices[code][0]
        sparsities.append((X == 0).mean())
    else:
        rep_dict = representations.get(code.split('_', 1)[0] + '_' + code.split('_', 1)[1] if '_' in code else code)
        if code in representations:
            rep_dict = representations[code]
            all_vecs = np.array([rep_dict[a] for a in amines_needed if a in rep_dict])
            sparsities.append((all_vecs == 0).mean())
        else:
            X = feature_matrices[code][0]
            amine_part = X[:, 1024:] if X.shape[1] > 1024 else X
            sparsities.append((amine_part == 0).mean())

bars = ax.bar(x, sparsities, width, color=plt.cm.Set2(np.linspace(0, 1, len(repr_codes))), 
              edgecolor='black', linewidth=0.5)
for i, s in enumerate(sparsities):
    ax.text(i, s + 0.02, f'{s:.1%}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Sparsity (fraction of zeros)', fontsize=11)
ax.set_title('Feature Sparsity by Representation', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Right: Dimensionality comparison
ax = axes[1]
dims = [feature_matrices[code][0].shape[1] for code in repr_codes]
bars = ax.bar(x, dims, width, color=plt.cm.Set2(np.linspace(0, 1, len(repr_codes))),
              edgecolor='black', linewidth=0.5)
for i, d in enumerate(dims):
    ax.text(i, d + 20, str(d), ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Total Feature Dimensions', fontsize=11)
ax.set_title('Feature Dimensionality by Representation', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Amine Representation Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(AMINE_REPR_DIR / 'amine_representation_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {AMINE_REPR_DIR / 'amine_representation_overview.png'}")

## 9. Save Results

In [ ]:
# Save full results (70 rows: 7 repr x 10 splits)
df_results.to_csv(AMINE_REPR_DIR / 'amine_repr_comparison_results.csv', index=False)
print(f"Saved: {AMINE_REPR_DIR / 'amine_repr_comparison_results.csv'} ({len(df_results)} rows)")

# Save summary table
df_summary.to_csv(AMINE_REPR_DIR / 'amine_repr_comparison_summary.csv', index=False)
print(f"Saved: {AMINE_REPR_DIR / 'amine_repr_comparison_summary.csv'} ({len(df_summary)} rows)")

# Save feature importance data
df_importance.to_csv(AMINE_REPR_DIR / 'amine_importance_fractions.csv', index=False)
print(f"Saved: {AMINE_REPR_DIR / 'amine_importance_fractions.csv'}")

print(f"\nAll outputs saved to: {AMINE_REPR_DIR}")

In [ ]:
# Final summary
print("=" * 80)
print("AMINE REPRESENTATION COMPARISON — FINAL SUMMARY")
print("=" * 80)
print(f"\nModel: Regularized XGBoost (max_depth=3, reg_alpha=1.0, reg_lambda=5.0)")
print(f"Enzyme: noncons_mean (conservation < 0.5, mean pooled, 1024-dim)")
print(f"Splits: {N_SPLITS} random enzyme hold-out splits")
print()

# Rank by PR-AUC
df_ranked = df_summary.sort_values('pr_auc_mean', ascending=False)
print(f"{'Rank':>4} {'Representation':<25} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12} {'LL Gap':>12}")
print("-" * 95)
for rank, (_, r) in enumerate(df_ranked.iterrows(), 1):
    marker = ' ***' if rank == 1 else ''
    print(f"{rank:>4} {r['representation']:<25} {r['total_dims']:>5.0f} "
          f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
          f"{r['pr_auc_mean']:.3f}+/-{r['pr_auc_std']:.3f} "
          f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f} "
          f"{r['logloss_gap_mean']:>+.3f}+/-{r['logloss_gap_std']:.3f}{marker}")

print()
best = df_ranked.iloc[0]
print(f"BEST REPRESENTATION: {best['representation']}")
print(f"  Total dims:  {best['total_dims']:.0f}")
print(f"  ROC-AUC:     {best['roc_auc_mean']:.3f} +/- {best['roc_auc_std']:.3f}")
print(f"  PR-AUC:      {best['pr_auc_mean']:.3f} +/- {best['pr_auc_std']:.3f}")
print(f"  F1:          {best['f1_mean']:.3f} +/- {best['f1_std']:.3f}")
print(f"  LogLoss gap: {best['logloss_gap_mean']:+.3f}")

print("\n" + "=" * 80)
print("KEY FINDINGS:")
print("-" * 80)

# Compare baseline (A) vs best
baseline = df_summary[df_summary['representation'] == 'A_morgan1024'].iloc[0]
print(f"\n1. Baseline (A: Morgan 1024):")
print(f"   ROC-AUC={baseline['roc_auc_mean']:.3f}, PR-AUC={baseline['pr_auc_mean']:.3f}, "
      f"LL gap={baseline['logloss_gap_mean']:+.3f}")
print(f"\n2. Best ({best['representation']}):")
print(f"   ROC-AUC={best['roc_auc_mean']:.3f}, PR-AUC={best['pr_auc_mean']:.3f}, "
      f"LL gap={best['logloss_gap_mean']:+.3f}")
print(f"\n3. ROC-AUC change: {best['roc_auc_mean'] - baseline['roc_auc_mean']:+.3f}")
print(f"   PR-AUC change:  {best['pr_auc_mean'] - baseline['pr_auc_mean']:+.3f}")
print(f"   LL gap change:  {best['logloss_gap_mean'] - baseline['logloss_gap_mean']:+.3f}")

# Feature importance insight
baseline_imp = df_importance[df_importance['representation'] == 'A_morgan1024'].iloc[0]
best_imp = df_importance[df_importance['representation'] == best['representation']].iloc[0]
print(f"\n4. Amine feature importance:")
print(f"   Baseline (A): {baseline_imp['amine_frac']:.1%} of total importance from amine features")
print(f"   Best:         {best_imp['amine_frac']:.1%} of total importance from amine features")

print("\n" + "=" * 80)